# ICS 591 Team 2 — Evaluation 1: the **baseline** on the local test set

**What the baseline is.** The Stage 1 ViT-S/16 trained only on RealWaste (Australian waste photos). It has never seen a
Ghanaian image. It predicts one of RealWaste's 9 material classes, so it can only answer **"plastic or not?"**.
It cannot say "dirty" and it cannot say "retake this photo".

**What this notebook measures on the 113 local test photos** (the same blind, human-agreed images used for v1):
1. **Plastic vs non-plastic** on the 76 photos that show waste (discards excluded): the question the baseline was built for.
2. **The two money errors** on all 113 (the baseline pays whenever it says "plastic"):
   - **wrong payments (FAR)**: photos that should not be paid but would be
   - **honest households missed (FRR)**: clean plastics that would not be paid
3. **The four app decisions**: plastic → accept, anything else → reject (wrong type). It can never output "dirty" or "retake".
4. **Trivial rules** (always non-plastic, most common decision), so the numbers have a floor to compare with.
5. Which RealWaste class it uses for Ghanaian photos, and example mistakes for the slides.

**How to run:** Runtime → Run all. No GPU needed. Results are saved to `MyDrive/ics591_evaluation/`.
Run this **before** notebook 09, which reads the baseline results for its comparison table.

In [ ]:
# 1. Setup: paths, the local TEST set, helpers. A GPU is optional (113 images also run on CPU in a few minutes).
!pip -q install timm
import os, json, shutil, pathlib, random, hashlib
import numpy as np, pandas as pd, torch, timm, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from torchvision import transforms as T
from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive')
LOCAL_OUT = DRIVE / 'ics591_local_data'             # Stage 2a/2b: images + label files
IMG_DRIVE = LOCAL_OUT / 'raw' / 'images'
S1_OUT = DRIVE / 'ics591_model_selection' / 'realwaste'   # Stage 1 outputs
STAGE1_CKPT = S1_OUT / 'ViT-S_16.pt'                 # baseline: Stage 1 ViT, RealWaste only
FT_OUT = DRIVE / 'ics591_finetune'                   # Stage 2c outputs
V1_CKPT, V1_CFG = FT_OUT / 'verifier_v1.pt', FT_OUT / 'verifier_v1_config.json'
EVAL_OUT = DRIVE / 'ics591_evaluation'; EVAL_OUT.mkdir(exist_ok=True)   # everything this notebook saves goes here
TIMM_ID = 'vit_small_patch16_224.augreg_in21k_ft_in1k'
for p in [IMG_DRIVE, LOCAL_OUT / 'al_queue.csv']: assert p.exists(), f'missing: {p}'

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Running on', DEV)

MATERIAL = ['plastic', 'non_plastic', 'discard']
CLEAN = ['clean', 'contaminated']
DECISIONS = ['accept', 'reject_contaminated', 'reject_wrong_type', 'retake_photo']
NICE = {'accept': 'Accept', 'reject_contaminated': 'Reject: dirty', 'reject_wrong_type': 'Reject: wrong type',
        'retake_photo': 'Retake'}

# --- label files (exactly the same logic as notebook 05, so the test set is the same 113 images)
TEAM = ['nana', 'kelvin', 'cajetan']
QUEUE, AUTO = LOCAL_OUT / 'al_queue.csv', LOCAL_OUT / 'al_auto_labels.csv'
def REV(p): return LOCAL_OUT / f'al_reviews_{p}.csv'
def to_model_label(l): return 'discard' if str(l).startswith('discard') else l
def load_reviews():
    parts = [pd.read_csv(REV(p)) for p in TEAM if REV(p).exists()]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['file', 'label', 'pool', 'labeler', 'ts'])
def test_files():
    q = pd.read_csv(QUEUE); return set(q.loc[q['pool'] == 'test', 'file'])
def test_labels():
    r = load_reviews(); r = r[r['pool'] == 'test'].sort_values('ts').drop_duplicates(['file', 'labeler'], keep='last')
    out = []
    for f, g in r.groupby('file'):
        vc = g['label'].map(to_model_label).value_counts()
        if len(vc) == 1 or vc.iloc[0] > vc.iloc[1]: out.append({'file': f, 'label': vc.index[0]})
    return pd.DataFrame(out, columns=['file', 'label'])
def current_labels():
    r = load_reviews(); r = r[r['pool'] != 'test'].sort_values('ts').drop_duplicates('file', keep='last')
    human = pd.DataFrame({'file': r['file'], 'label': r['label'].map(to_model_label), 'source': 'human'})
    auto = pd.read_csv(AUTO) if AUTO.exists() else pd.DataFrame(columns=['file', 'label'])
    auto = pd.DataFrame({'file': auto['file'], 'label': auto['label'], 'source': 'model'})
    auto = auto[~auto['file'].isin(human['file'])]
    lab = pd.concat([human, auto], ignore_index=True)
    return lab[~lab['file'].isin(test_files())].reset_index(drop=True)
LABEL_TO_DECISION = {'plastic_clean': 0, 'plastic_contaminated': 1, 'non_plastic': 2, 'discard': 3}
LABEL_TO_MATERIAL = {'plastic_clean': 0, 'plastic_contaminated': 0, 'non_plastic': 1, 'discard': 2}

loc_te = test_labels().sort_values('file').reset_index(drop=True)
print('Local TEST set:', len(loc_te), 'images', loc_te['label'].value_counts().to_dict())
if len(loc_te) != 113: print('WARNING: expected 113 test images (as in Stage 2c). Have label files changed since?')
y_dec = loc_te['label'].map(LABEL_TO_DECISION).values     # true decision: 0 accept, 1 dirty, 2 wrong type, 3 retake
y_mat = loc_te['label'].map(LABEL_TO_MATERIAL).values     # true material: 0 plastic, 1 non-plastic, 2 discard

CACHE = pathlib.Path('/content/img_cache'); CACHE.mkdir(exist_ok=True)
for f in loc_te['file']:
    if not (CACHE / f).exists(): shutil.copy(IMG_DRIVE / f, CACHE / f)

_cfg = timm.data.resolve_data_config({}, model=timm.create_model(TIMM_ID, pretrained=False))
EVAL_TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(_cfg['mean'], _cfg['std'])])

@torch.no_grad()
def run(model, files, batch=32):
    """Runs a model over image files; returns a list with one numpy array per model output."""
    model.eval(); outs = None
    for i in range(0, len(files), batch):
        x = torch.stack([EVAL_TF(Image.open(CACHE / f).convert('RGB')) for f in files[i:i + batch]]).to(DEV)
        o = model(x); o = o if isinstance(o, tuple) else (o,)
        o = [t.float().softmax(1).cpu().numpy() for t in o]
        outs = o if outs is None else [np.vstack([a, b]) for a, b in zip(outs, o)]
    return outs

def ci(correct, n_boot=2000):
    """95% bootstrap interval for a mean of 0/1 values (resample the test images 2000 times)."""
    v = np.asarray(correct, dtype=float); rng = np.random.default_rng(SEED)
    b = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n_boot)]
    return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]
def pct(x): return f'{100 * x:.1f}%'
def show_examples(idx, title, captions, path, cols=5):
    """Picture grid for slides."""
    idx = list(idx)[:15]
    if not idx: print('no examples'); return
    rows = int(np.ceil(len(idx) / cols)); fig, axs = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.6 * rows), squeeze=False)
    for a in axs.flat: a.axis('off')
    for a, i in zip(axs.flat, idx):
        a.imshow(Image.open(CACHE / loc_te['file'][i]).convert('RGB')); a.set_title(captions[i], fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=130); plt.show()

In [ ]:
# 2. Load the baseline and run it on the 113 test photos
cc = S1_OUT / 'split_counts.csv'
RW_CLASSES = [c for c in pd.read_csv(cc, index_col=0).index if c != 'TOTAL'] if cc.exists() else \
    ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
PLASTIC = next(i for i, c in enumerate(RW_CLASSES) if 'plastic' in c.lower())
base = timm.create_model(TIMM_ID, pretrained=False, num_classes=len(RW_CLASSES))
base.load_state_dict(torch.load(STAGE1_CKPT, map_location='cpu')); base = base.to(DEV)
(P,) = run(base, list(loc_te['file']))
rw_pred = P.argmax(1)                         # which of the 9 RealWaste classes
says_plastic = rw_pred == PLASTIC
p_plastic = P[:, PLASTIC]
print('RealWaste classes:', RW_CLASSES, '| plastic index', PLASTIC)
print('Baseline says "plastic" for', says_plastic.sum(), 'of', len(says_plastic), 'photos')

In [ ]:
# 3. Results
R = {'model': 'baseline: Stage 1 ViT-S/16, RealWaste only', 'local_test_n': int(len(loc_te))}

# (a) plastic vs non-plastic, waste photos only (discards excluded) -- the question the baseline was trained for
w = y_mat != 2
ok_a = (says_plastic[w] == (y_mat[w] == 0))
R['plastic_vs_non_n'] = int(w.sum())
R['plastic_vs_non_accuracy'] = float(ok_a.mean()); R['plastic_vs_non_accuracy_95ci'] = ci(ok_a)
tp = int((says_plastic & w & (y_mat == 0)).sum()); fp = int((says_plastic & w & (y_mat == 1)).sum())
R['plastic_recall'] = tp / max(1, int((w & (y_mat == 0)).sum()))
R['plastic_precision'] = tp / max(1, tp + fp)

# (b) the two money errors, all 113 photos -- pays when it says plastic (it cannot see dirt)
should_pay = y_dec == 0
R['wrong_payments'] = int((says_plastic & ~should_pay).sum())
R['false_accept_rate'] = float((says_plastic & ~should_pay).sum() / (~should_pay).sum())
R['honest_missed'] = int((~says_plastic & should_pay).sum())
R['false_reject_rate'] = float((~says_plastic & should_pay).sum() / should_pay.sum())
R['wrong_payments_by_true_label'] = loc_te['label'][says_plastic & ~should_pay].value_counts().to_dict()

# (c) the four app decisions
dec_b = np.where(says_plastic, 0, 2)
ok_c = dec_b == y_dec
from sklearn.metrics import f1_score
R['decision_accuracy_4way'] = float(ok_c.mean()); R['decision_accuracy_4way_95ci'] = ci(ok_c)
R['decision_macro_f1_4way'] = float(f1_score(y_dec, dec_b, average='macro', labels=range(4), zero_division=0))

# (d) trivial rules (the floor)
R['trivial_always_non_plastic_on_waste_photos'] = float((y_mat[w] == 1).mean())
R['trivial_majority_decision_4way'] = float(pd.Series(y_dec).value_counts(normalize=True).iloc[0])

(EVAL_OUT / 'baseline_results.json').write_text(json.dumps(R, indent=2))
pd.DataFrame({'file': loc_te['file'], 'true_label': loc_te['label'], 'realwaste_class': [RW_CLASSES[i] for i in rw_pred],
              'p_plastic': p_plastic.round(4), 'baseline_decision': [DECISIONS[d] for d in dec_b]}
             ).to_csv(EVAL_OUT / 'baseline_predictions.csv', index=False)

summary = pd.DataFrame([
    ['Plastic vs non-plastic (waste photos)', f"{pct(R['plastic_vs_non_accuracy'])}  (95% CI {pct(R['plastic_vs_non_accuracy_95ci'][0])}-{pct(R['plastic_vs_non_accuracy_95ci'][1])})", f"n = {R['plastic_vs_non_n']}"],
    ['Wrong payments, FAR', f"{R['wrong_payments']} of {int((~should_pay).sum())}  ({pct(R['false_accept_rate'])})", str(R['wrong_payments_by_true_label'])],
    ['Honest households missed, FRR', f"{R['honest_missed']} of {int(should_pay.sum())}  ({pct(R['false_reject_rate'])})", 'clean plastic not paid'],
    ['Four app decisions correct', pct(R['decision_accuracy_4way']), 'can never say "dirty" or "retake"'],
    ['Trivial: always non-plastic', pct(R['trivial_always_non_plastic_on_waste_photos']), 'floor for row 1'],
    ['Trivial: most common decision', pct(R['trivial_majority_decision_4way']), 'floor for row 4'],
], columns=['Measure', 'Baseline', 'Note'])
summary.to_csv(EVAL_OUT / 'baseline_summary_table.csv', index=False)
print('Stage 2c reported 72.4% for plastic vs non-plastic -- this run:', pct(R['plastic_vs_non_accuracy']))
summary

In [ ]:
# 4. What does an Australian-trained model call Ghanaian waste? (rows: our label, columns: RealWaste class it chose)
ct = pd.crosstab(loc_te['label'], pd.Series([RW_CLASSES[i] for i in rw_pred], name='baseline says'))
ct.to_csv(EVAL_OUT / 'baseline_realwaste_class_crosstab.csv'); ct

In [ ]:
# 5. Example mistakes for the slides: photos it would have PAID for but should not, then clean plastics it missed
cap = {i: f"ours: {loc_te['label'][i]}\nbaseline: {RW_CLASSES[rw_pred[i]]} ({P[i].max():.2f})" for i in range(len(loc_te))}
show_examples(np.where(says_plastic & ~should_pay)[0], 'Baseline would pay, but should not', cap, EVAL_OUT / 'baseline_wrong_payments.png')
show_examples(np.where(~says_plastic & should_pay)[0], 'Clean plastic the baseline missed', cap, EVAL_OUT / 'baseline_missed_clean.png')

## How to read this
- **Row 1 is the baseline's best case**: the only question it was trained for. Compare it with "always non-plastic".
- **Rows 2–3 are the money errors.** FAR loses money; FRR leaves an honest household unpaid. The baseline cannot see dirt,
  so any dirty plastic it recognises as plastic becomes a wrong payment.
- **Row 4**: it can never be right on "dirty" or "retake" photos, so its four-way score is capped.
- Everything is saved in `MyDrive/ics591_evaluation/` (`baseline_results.json`, tables and pictures). Next: notebook 09.